In [1]:
import numpy as np 
import pandas as pd 

dataset = pd.read_csv('cicids2017_cleaned.csv')

In [2]:
dataset.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Max,Active Min,Idle Mean,Idle Max,Idle Min,Attack Type
0,22,1266342,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
1,22,1319353,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
2,22,160,1,0,0,0,0.000000,0.000000,0,0,...,243,0,32,0.0,0,0,0.0,0,0,Normal Traffic
3,22,1303488,41,2728,456,0,66.536585,110.129945,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
4,35396,77,1,0,0,0,0.000000,0.000000,0,0,...,290,0,32,0.0,0,0,0.0,0,0,Normal Traffic


In [3]:
n_rows, n_cols = dataset.shape
print(f"Rows: {n_rows}, Columns: {n_cols}")

dataset.dtypes



Rows: 2520751, Columns: 53


Destination Port                 int64
Flow Duration                    int64
Total Fwd Packets                int64
Total Length of Fwd Packets      int64
Fwd Packet Length Max            int64
Fwd Packet Length Min            int64
Fwd Packet Length Mean         float64
Fwd Packet Length Std          float64
Bwd Packet Length Max            int64
Bwd Packet Length Min            int64
Bwd Packet Length Mean         float64
Bwd Packet Length Std          float64
Flow Bytes/s                   float64
Flow Packets/s                 float64
Flow IAT Mean                  float64
Flow IAT Std                   float64
Flow IAT Max                     int64
Flow IAT Min                     int64
Fwd IAT Total                    int64
Fwd IAT Mean                   float64
Fwd IAT Std                    float64
Fwd IAT Max                      int64
Fwd IAT Min                      int64
Bwd IAT Total                    int64
Bwd IAT Mean                   float64
Bwd IAT Std              

In [4]:
dataset['Attack Type'].value_counts()


Attack Type
Normal Traffic    2095057
DoS                193745
DDoS               128014
Port Scanning       90694
Brute Force          9150
Web Attacks          2143
Bots                 1948
Name: count, dtype: int64

In [5]:
#Mapping the 7 classes to the new 3

def map_attack_type(label):
    if label == "Normal Traffic":
        return "Normal"
    elif label in ["DoS", "DDoS"]:
        return "DoS"
    else:
        return "Other"

dataset["Attack_Class"] = dataset["Attack Type"].apply(map_attack_type)


In [6]:
class_counts = dataset["Attack_Class"].value_counts()
class_percent = dataset["Attack_Class"].value_counts(normalize=True) * 100

print(class_counts)
print(class_percent)


Attack_Class
Normal    2095057
DoS        321759
Other      103935
Name: count, dtype: int64
Attack_Class
Normal    83.112414
DoS       12.764410
Other      4.123176
Name: proportion, dtype: float64


In [7]:
#Data cleaning

n_nan = dataset.isna().sum().sum()
n_inf = np.isinf(dataset.select_dtypes(include=[np.number])).sum().sum()

print("NaN values:", n_nan)
print("Infinite values:", n_inf)


NaN values: 0
Infinite values: 0


In [8]:
#Switch to Attack_Class instead of Attack Type
dataset = dataset.drop(columns=["Attack Type"])
X = dataset.drop(columns=["Attack_Class"])
y = dataset["Attack_Class"]

# Encode labes to work with RF

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

label_encoder.classes_


array(['DoS', 'Normal', 'Other'], dtype=object)

In [9]:
# Conduct stratified train/test split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)

In [10]:
unique, counts = np.unique(y_train, return_counts=True)
print("Train distribution:", dict(zip(unique, counts)))

unique, counts = np.unique(y_test, return_counts=True)
print("Test distribution:", dict(zip(unique, counts)))

Train distribution: {np.int64(0): np.int64(257407), np.int64(1): np.int64(1676045), np.int64(2): np.int64(83148)}
Test distribution: {np.int64(0): np.int64(64352), np.int64(1): np.int64(419012), np.int64(2): np.int64(20787)}


In [11]:
# Baseline RF
from sklearn.ensemble import RandomForestClassifier

rf_baseline = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_baseline.fit(X_train, y_train)


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [12]:
y_pred = rf_baseline.predict(X_test)

# Fetch metrics from baseline RF

from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    digits=4
))


              precision    recall  f1-score   support

         DoS     0.9986    0.9980    0.9983     64352
      Normal     0.9990    0.9993    0.9992    419012
       Other     0.9891    0.9865    0.9878     20787

    accuracy                         0.9986    504151
   macro avg     0.9956    0.9946    0.9951    504151
weighted avg     0.9986    0.9986    0.9986    504151



In [13]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)


[[ 64223    126      3]
 [    81 418707    224]
 [     6    275  20506]]


In [14]:
# Adding improvments 

depths = [None, 10, 20, 30]

for d in depths:
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=d,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    print(f"\nmax_depth = {d}")
    print(classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        digits=4
    ))


| max_depth | Other Recall |   Other F1 |
| --------: | -----------: | ---------: |
|      None |       0.9865 |     0.9878 |
|        10 |       0.9916 |     0.9864 |
|    **20** |   **0.9978** | **0.9904** |
|        30 |       0.9888 |     0.9874 |

Conclusion:
- Using depth 20 improves recall for "Other" class
- It comes to no meaningful cost to DoS or Normal traffic
- Accuracy increases slightly 

With a depth of 20 for the *other class* **precision** was lower meaning less false positives. While **recall** was increased meaning less missed attacks. Which is deseriable charactericts in the NIDS. 

Depth 10 was worst because the trees were comparitvly shallow which decraces the models performance. 

In [15]:
#Feature importance

rf_final = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_final.fit(X_train, y_train)


,n_estimators,100
,criterion,'gini'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [16]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_final.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importance.head(15)


,feature,importance
36,Packet Length Variance,0.081425
11,Bwd Packet Length Std,0.073098
35,Packet Length Std,0.065933
34,Packet Length Mean,0.062393
40,Average Packet Size,0.061072
4,Fwd Packet Length Max,0.052057
10,Bwd Packet Length Mean,0.051741
41,Subflow Fwd Bytes,0.046594
33,Max Packet Length,0.043552
3,Total Length of Fwd Packets,0.038839


In [19]:
#Setting up report deliverabes 
from sklearn.metrics import classification_report, accuracy_score

# Baseline predictions
y_pred_baseline = rf_baseline.predict(X_test)

# Classification report as dict
report_baseline = classification_report(
    y_test,
    y_pred_baseline,
    target_names=label_encoder.classes_,
    output_dict=True
)

acc_baseline = accuracy_score(y_test, y_pred_baseline)


In [20]:
# Final model predictions
y_pred_final = rf_final.predict(X_test)

# Classification report as dict
report_final = classification_report(
    y_test,
    y_pred_final,
    target_names=label_encoder.classes_,
    output_dict=True
)

acc_final = accuracy_score(y_test, y_pred_final)


In [21]:
results_table = pd.DataFrame({
    "Model": ["Baseline RF", "Tuned RF (max_depth=20)"],
    "DoS F1": [
        report_baseline["DoS"]["f1-score"],
        report_final["DoS"]["f1-score"]
    ],
    "Normal F1": [
        report_baseline["Normal"]["f1-score"],
        report_final["Normal"]["f1-score"]
    ],
    "Other F1": [
        report_baseline["Other"]["f1-score"],
        report_final["Other"]["f1-score"]
    ],
    "Accuracy": [
        acc_baseline,
        acc_final
    ]
})

results_table


,Model,DoS F1,Normal F1,Other F1,Accuracy
0,Baseline RF,0.998321,0.999158,0.987765,0.998582
1,Tuned RF (max_depth=20),0.998471,0.999304,0.990355,0.998826


In [22]:
results_table.round(4)


,Model,DoS F1,Normal F1,Other F1,Accuracy
0,Baseline RF,0.9983,0.9992,0.9878,0.9986
1,Tuned RF (max_depth=20),0.9985,0.9993,0.9904,0.9988
